# 📐 Module 02 — JSON Schema & Structured LLM Outputs
## Make LLMs Reliably Structured

> **Marevlo AI Platform** · All Levels: Beginner → Expert

---
### What you'll learn
- What JSON Schema is and how to write schemas from scratch
- How to validate JSON in Python using `jsonschema`
- How to force structured outputs from Claude and OpenAI using tool schemas
- How to use Pydantic to auto-generate schemas and validate LLM responses
- How to use the `instructor` library for one-line structured extraction
- Production patterns: retry loops, schema versioning, middleware validation

---

In [ ]:
!pip install jsonschema pydantic instructor anthropic -q

---
## 📋 Part 1 — JSON Schema Fundamentals
### 🟢 Beginner: Writing Your First Schema

In [ ]:
import json
from jsonschema import validate, ValidationError, Draft7Validator

# === Your first JSON Schema ===
device_report_schema = {
    "type": "object",
    "description": "Analysis report for a network device",
    "properties": {
        "device_id": {
            "type": "string",
            "description": "Device identifier",
            "minLength": 1,
            "maxLength": 20
        },
        "risk_level": {
            "type": "string",
            "enum": ["LOW", "MED", "HIGH"],
            "description": "Assessed risk level"
        },
        "anomaly_score": {
            "type": "number",
            "minimum": 0,
            "maximum": 1,
            "description": "0=normal, 1=anomalous"
        },
        "affected_components": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "uniqueItems": True
        },
        "timestamp": {
            "type": "integer",
            "description": "Unix epoch seconds"
        }
    },
    "required": ["device_id", "risk_level", "anomaly_score"],
    "additionalProperties": False   # Reject unexpected keys
}

print("Schema structure:")
print(json.dumps(device_report_schema, indent=2))

In [ ]:
# === Test valid data ===
valid_data = {
    "device_id": "JNP-001",
    "risk_level": "HIGH",
    "anomaly_score": 0.91,
    "affected_components": ["CPU", "PFE"]
}

validate(instance=valid_data, schema=device_report_schema)
print("✓ Valid data passed schema validation")

In [ ]:
# === Test invalid data — various failures ===
test_cases = [
    {
        "name": "Invalid enum value",
        "data": {"device_id": "JNP-001", "risk_level": "CRITICAL", "anomaly_score": 0.9}
    },
    {
        "name": "Score out of range",
        "data": {"device_id": "JNP-001", "risk_level": "HIGH", "anomaly_score": 1.5}
    },
    {
        "name": "Missing required field",
        "data": {"device_id": "JNP-001", "anomaly_score": 0.9}  # No risk_level
    },
    {
        "name": "Additional properties",
        "data": {"device_id": "JNP-001", "risk_level": "LOW", "anomaly_score": 0.1, "extra_field": "oops"}
    }
]

for tc in test_cases:
    try:
        validate(instance=tc["data"], schema=device_report_schema)
        print(f"✓ {tc['name']}: passed (unexpected!)")
    except ValidationError as e:
        print(f"✗ {tc['name']}: {e.message}")

### 🟡 Intermediate: Collecting All Errors + $ref

In [ ]:
# Always use Draft7Validator.iter_errors() in production
# It collects ALL validation errors, not just the first one

validator = Draft7Validator(device_report_schema)

# Data with multiple simultaneous errors
bad_llm_output = {
    "device_id": "",            # minLength violation (empty string)
    "risk_level": "EXTREME",   # enum violation
    "anomaly_score": 1.5,       # maximum violation
    "extra_key": "hallucinated" # additionalProperties violation
}

errors = list(validator.iter_errors(bad_llm_output))
print(f"Found {len(errors)} validation errors:\n")

for i, error in enumerate(errors, 1):
    # Build path string like "properties.risk_level"
    path = " > ".join(str(p) for p in error.absolute_path) or "root"
    print(f"  Error {i} [{path}]: {error.message}")

# Build a retry prompt from errors
error_summary = "\n".join(
    f"- {' > '.join(str(p) for p in e.absolute_path) or 'root'}: {e.message}"
    for e in errors
)
retry_prompt = f"""
Your previous response had {len(errors)} validation errors:
{error_summary}

Please fix all errors and return a valid JSON object.
"""
print(f"\nRetry prompt:\n{retry_prompt}")

In [ ]:
# $ref — Reusable sub-schemas
schema_with_refs = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "definitions": {
        "Severity": {
            "type": "string",
            "enum": ["LOW", "MED", "HIGH", "CRITICAL"]
        },
        "Device": {
            "type": "object",
            "properties": {
                "id": {"type": "string"},
                "hostname": {"type": "string"}
            },
            "required": ["id"]
        }
    },
    "type": "object",
    "properties": {
        "primary_device": {"$ref": "#/definitions/Device"},
        "related_devices": {
            "type": "array",
            "items": {"$ref": "#/definitions/Device"}
        },
        "max_severity": {"$ref": "#/definitions/Severity"}
    },
    "required": ["primary_device", "max_severity"]
}

test_data = {
    "primary_device": {"id": "JNP-001", "hostname": "edge-router-01"},
    "related_devices": [{"id": "JNP-002"}, {"id": "JNP-003"}],
    "max_severity": "HIGH"
}

validate(instance=test_data, schema=schema_with_refs)
print("✓ $ref schema validated successfully")

### 🔴 Advanced: Conditional Schemas + oneOf

In [ ]:
# if/then/else — Conditional validation
# Rule: if risk_level is HIGH, escalation_contact is required

conditional_schema = {
    "type": "object",
    "properties": {
        "risk_level": {"type": "string", "enum": ["LOW", "MED", "HIGH"]},
        "escalation_contact": {"type": "string"},
        "device_id": {"type": "string"}
    },
    "required": ["risk_level", "device_id"],
    "if": {
        "properties": {"risk_level": {"const": "HIGH"}},
        "required": ["risk_level"]
    },
    "then": {
        "required": ["escalation_contact"]
    }
}

validator = Draft7Validator(conditional_schema)

# Case 1: LOW risk — no escalation needed
low_risk = {"risk_level": "LOW", "device_id": "JNP-001"}
errors = list(validator.iter_errors(low_risk))
print(f"LOW risk without contact: {'✓ Valid' if not errors else f'✗ {errors[0].message}'}")

# Case 2: HIGH risk WITHOUT escalation — should fail
high_no_contact = {"risk_level": "HIGH", "device_id": "JNP-001"}
errors = list(validator.iter_errors(high_no_contact))
print(f"HIGH risk without contact: {'✓ Valid' if not errors else f'✗ {errors[0].message}'}")

# Case 3: HIGH risk WITH escalation — should pass
high_with_contact = {"risk_level": "HIGH", "device_id": "JNP-001", "escalation_contact": "oncall@marevlo.com"}
errors = list(validator.iter_errors(high_with_contact))
print(f"HIGH risk with contact: {'✓ Valid' if not errors else f'✗ {errors[0].message}'}")

In [ ]:
# oneOf — Polymorphic agent actions
action_schema = {
    "type": "object",
    "properties": {
        "action": {
            "oneOf": [
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "alert"},
                        "channel": {"type": "string", "enum": ["slack", "email", "pagerduty"]},
                        "message": {"type": "string", "minLength": 5}
                    },
                    "required": ["type", "channel", "message"],
                    "additionalProperties": False
                },
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "restart_service"},
                        "service_name": {"type": "string"},
                        "confirm": {"type": "boolean", "const": True}
                    },
                    "required": ["type", "service_name", "confirm"],
                    "additionalProperties": False
                }
            ]
        }
    },
    "required": ["action"]
}

v = Draft7Validator(action_schema)

alert_action = {"action": {"type": "alert", "channel": "slack", "message": "JNP-001 CPU critical"}}
restart_action = {"action": {"type": "restart_service", "service_name": "rpd", "confirm": True}}
invalid_action = {"action": {"type": "unknown_action", "foo": "bar"}}

for name, data in [("Alert action", alert_action), ("Restart action", restart_action), ("Invalid action", invalid_action)]:
    errors = list(v.iter_errors(data))
    status = '✓ Valid' if not errors else f'✗ Invalid ({len(errors)} errors)'
    print(f"{name}: {status}")

---
## 🤖 Part 2 — Structured LLM Outputs
### 🟢 Beginner: Prompt Engineering vs Tool Use

In [ ]:
# Demonstrate the problem with naive prompt-only approach
import json

# Simulating what an LLM might return with prompt-only approach
llm_responses_examples = [
    # Good response
    '{"device_id": "JNP-001", "risk_level": "HIGH", "anomaly_score": 0.91}',
    # Markdown-wrapped
    '```json\n{"device_id": "JNP-001", "risk_level": "HIGH", "anomaly_score": 0.91}\n```',
    # With explanation prefix
    'Here is the analysis:\n{"device_id": "JNP-001", "risk_level": "HIGH", "anomaly_score": 0.91}',
    # Wrong field name
    '{"device": "JNP-001", "severity": "HIGH", "score": 0.91}',
    # Out-of-enum value
    '{"device_id": "JNP-001", "risk_level": "CRITICAL", "anomaly_score": 0.91}',
]

def safe_parse_llm_json(text: str) -> dict | None:
    """Attempt to parse LLM output as JSON, handling common issues."""
    text = text.strip()
    # Strip markdown fences
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(lines[1:-1]).strip()
    # Extract JSON from text with prefix
    start = text.find('{')
    if start > 0:
        text = text[start:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print("Parsing results from different LLM response formats:")
for i, response in enumerate(llm_responses_examples, 1):
    parsed = safe_parse_llm_json(response)
    status = '✓ Parsed' if parsed else '✗ Failed'
    print(f"  Response {i}: {status}")
    if parsed:
        print(f"    risk_level: {parsed.get('risk_level', 'MISSING')}")

### 🟡 Intermediate: Anthropic Tool Use for Structured Output

In [ ]:
# Define a tool that IS your output schema
import anthropic

output_tool = {
    "name": "report_device_analysis",
    "description": "Report the structured findings from device analysis",
    "input_schema": {
        "type": "object",
        "properties": {
            "device_id": {
                "type": "string",
                "description": "The device being analyzed (e.g. JNP-001)"
            },
            "risk_level": {
                "type": "string",
                "enum": ["LOW", "MED", "HIGH"],
                "description": "LOW=monitor only, MED=schedule maintenance, HIGH=immediate action required"
            },
            "anomaly_score": {
                "type": "number",
                "minimum": 0,
                "maximum": 1,
                "description": "0.0 = completely normal operation, 1.0 = definite failure imminent"
            },
            "root_cause": {
                "type": "string",
                "description": "The single most likely root cause in one clear sentence"
            },
            "recommended_actions": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": "Concrete remediation steps ordered by priority"
            }
        },
        "required": ["device_id", "risk_level", "anomaly_score", "recommended_actions"]
    }
}

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    tools=[output_tool],
    tool_choice={"type": "tool", "name": "report_device_analysis"},  # Force this tool
    messages=[{
        "role": "user",
        "content": "Analyze device JNP-001: CPU utilization=94.2%, memory=67.3%, PFE errors=412 in last hour. Is this anomalous?"
    }]
)

# Extract structured output
tool_block = next(b for b in response.content if b.type == "tool_use")
report = tool_block.input

print(f"Device: {report['device_id']}")
print(f"Risk: {report['risk_level']}")
print(f"Score: {report['anomaly_score']:.2f}")
print(f"Root cause: {report.get('root_cause', 'N/A')}")
print("\nRecommended actions:")
for action in report['recommended_actions']:
    print(f"  • {action}")

---
## 🐍 Part 3 — Pydantic Integration
### 🟢 Beginner: Auto Schema Generation

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Literal, Optional
import json

# Define your output shape as a Python class
class DeviceReport(BaseModel):
    """Analysis report for a network device."""
    device_id: str = Field(description="Device identifier e.g. JNP-001")
    risk_level: Literal["LOW", "MED", "HIGH"]
    anomaly_score: float = Field(ge=0.0, le=1.0, description="0=normal, 1=anomalous")
    root_cause: Optional[str] = Field(default=None)
    recommended_actions: list[str] = Field(min_length=1)

# Auto-generate JSON Schema
schema = DeviceReport.model_json_schema()
print("Auto-generated JSON Schema:")
print(json.dumps(schema, indent=2))

In [ ]:
# Validate by construction
try:
    report = DeviceReport(
        device_id="JNP-001",
        risk_level="HIGH",
        anomaly_score=0.91,
        recommended_actions=["Check PFE logs", "Restart routing daemon"]
    )
    print(f"✓ Valid report: {report.device_id} → {report.risk_level}")
    print(f"  JSON: {report.model_dump_json()}")
except Exception as e:
    print(f"✗ Validation error: {e}")

# Test invalid data
test_invalid = [
    {"device_id": "JNP-001", "risk_level": "EXTREME", "anomaly_score": 0.9, "recommended_actions": ["check"]},
    {"device_id": "JNP-001", "risk_level": "HIGH", "anomaly_score": 1.5, "recommended_actions": ["check"]},
    {"device_id": "JNP-001", "risk_level": "HIGH", "anomaly_score": 0.9, "recommended_actions": []},
]

print("\nInvalid data tests:")
for data in test_invalid:
    try:
        DeviceReport(**data)
        print(f"  UNEXPECTED PASS for {data}")
    except Exception as e:
        first_err = e.errors()[0]
        print(f"  ✗ Caught: {first_err['loc']} → {first_err['msg']}")

### 🟡 Intermediate: Pydantic → Anthropic Tool Schema

In [ ]:
import anthropic

class NetworkAlert(BaseModel):
    """Alert generated when network anomaly is detected."""
    device_id: str = Field(description="Affected device ID")
    severity: Literal["LOW", "MED", "HIGH", "CRITICAL"]
    metric: str = Field(description="The metric that triggered the alert")
    current_value: float
    threshold: float
    suggested_action: str
    notify_oncall: bool = Field(default=False, description="Page the on-call engineer")

def pydantic_to_anthropic_tool(model: type[BaseModel], tool_name: str) -> dict:
    """Convert a Pydantic model into an Anthropic tool definition."""
    schema = model.model_json_schema()
    # Clean up Pydantic-specific meta fields
    schema.pop("$defs", None)
    schema.pop("title", None)
    return {
        "name": tool_name,
        "description": model.__doc__ or f"Call {tool_name}",
        "input_schema": schema
    }

# Build the tool
alert_tool = pydantic_to_anthropic_tool(NetworkAlert, "send_network_alert")
print("Generated tool definition:")
print(json.dumps(alert_tool, indent=2))

In [ ]:
# Use the tool with Anthropic API and parse into Pydantic model
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=512,
    tools=[alert_tool],
    tool_choice={"type": "tool", "name": "send_network_alert"},
    messages=[{
        "role": "user",
        "content": "Device JNP-001: CPU utilization is 94.2%, threshold is 80%. Create a network alert."
    }]
)

# Deserialize directly into Pydantic model
tool_input = response.content[0].input
alert = NetworkAlert.model_validate(tool_input)

print(f"Alert created:")
print(f"  Device: {alert.device_id}")
print(f"  Severity: {alert.severity}")
print(f"  Metric: {alert.metric}")
print(f"  Value: {alert.current_value} (threshold: {alert.threshold})")
print(f"  Notify on-call: {alert.notify_oncall}")
print(f"  Action: {alert.suggested_action}")

### 🔴 Advanced: Complex Nested Models with Cross-field Validation

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Literal, Optional
from datetime import datetime
import re

class Metric(BaseModel):
    name: str
    value: float
    unit: str
    is_anomalous: bool

class RemediationStep(BaseModel):
    order: int = Field(ge=1)
    action: str = Field(min_length=10, description="Clear description of the action")
    estimated_minutes: int = Field(ge=1, le=480)
    requires_downtime: bool

class ComprehensiveReport(BaseModel):
    device_id: str
    risk_level: Literal["LOW", "MED", "HIGH"]
    anomaly_score: float = Field(ge=0, le=1)
    metrics: list[Metric]
    root_cause_hypothesis: str
    remediation_steps: list[RemediationStep]
    
    @field_validator("device_id")
    @classmethod
    def validate_device_id_format(cls, v: str) -> str:
        if not re.match(r"^[A-Z]{2,4}-\d{3}$", v):
            raise ValueError(f"Invalid device ID format: '{v}'. Expected format: JNP-001")
        return v
    
    @model_validator(mode="after")
    def high_risk_requires_steps(self):
        if self.risk_level == "HIGH" and len(self.remediation_steps) < 2:
            raise ValueError("HIGH risk devices require at least 2 remediation steps")
        return self
    
    @model_validator(mode="after")
    def score_aligns_with_risk(self):
        thresholds = {"LOW": 0.5, "MED": 0.7, "HIGH": 0.8}
        min_score = thresholds[self.risk_level]
        if self.risk_level == "HIGH" and self.anomaly_score < min_score:
            raise ValueError(f"HIGH risk should have anomaly_score >= {min_score}, got {self.anomaly_score}")
        return self

# Test valid complex report
valid_report = ComprehensiveReport(
    device_id="JNP-001",
    risk_level="HIGH",
    anomaly_score=0.91,
    metrics=[
        Metric(name="cpu_util", value=94.2, unit="percent", is_anomalous=True),
        Metric(name="mem_util", value=67.3, unit="percent", is_anomalous=False),
    ],
    root_cause_hypothesis="Excessive BGP route churn causing CPU saturation in the routing daemon",
    remediation_steps=[
        RemediationStep(order=1, action="Restart the routing process daemon (rpd)", estimated_minutes=5, requires_downtime=False),
        RemediationStep(order=2, action="Investigate BGP peer flapping on upstream links", estimated_minutes=30, requires_downtime=False),
    ]
)
print(f"✓ Valid: {valid_report.device_id} ({valid_report.risk_level})")

# Test cross-field validation
try:
    ComprehensiveReport(
        device_id="JNP-001",
        risk_level="HIGH",
        anomaly_score=0.3,  # Too low for HIGH risk!
        metrics=[Metric(name="cpu_util", value=30.0, unit="percent", is_anomalous=False)],
        root_cause_hypothesis="Minor issue",
        remediation_steps=[
            RemediationStep(order=1, action="Monitor the device closely", estimated_minutes=60, requires_downtime=False),
            RemediationStep(order=2, action="Schedule maintenance window", estimated_minutes=120, requires_downtime=True),
        ]
    )
except Exception as e:
    print(f"✗ Cross-field validation caught: {e.errors()[0]['msg']}")

---
## ⚡ Part 4 — Instructor Library
### 🟢 Beginner: One-Line Structured Extraction

In [ ]:
import instructor
import anthropic
from pydantic import BaseModel, Field
from typing import Literal

class DeviceAnalysis(BaseModel):
    """Structured analysis of a network device."""
    device_id: str
    risk_level: Literal["LOW", "MED", "HIGH"] = Field(
        description="LOW=monitor, MED=schedule maintenance, HIGH=immediate action"
    )
    anomaly_score: float = Field(
        ge=0, le=1,
        description="0.0=completely normal, 1.0=definite failure imminent"
    )
    summary: str = Field(description="One paragraph summary of findings")
    top_3_actions: list[str] = Field(min_length=1, max_length=3)

# Patch the Anthropic client with instructor
client = instructor.from_anthropic(anthropic.Anthropic())

# One call — returns a validated DeviceAnalysis object!
analysis: DeviceAnalysis = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    response_model=DeviceAnalysis,
    messages=[{
        "role": "user",
        "content": """Analyze this device telemetry:
Device: JNP-001
CPU utilization: 94.2% (5-min avg, threshold: 80%)
Memory utilization: 67.3%
PFE errors last hour: 412 (baseline: <10)
BGP flaps last hour: 8
Uptime: 180 days"""
    }]
)

# analysis is a validated Python object with type safety
print(f"Device: {analysis.device_id}")
print(f"Risk: {analysis.risk_level}")
print(f"Score: {analysis.anomaly_score:.2f}")
print(f"\nSummary: {analysis.summary[:100]}...")
print("\nTop actions:")
for i, action in enumerate(analysis.top_3_actions, 1):
    print(f"  {i}. {action}")

---
## 🏭 Part 5 — Production Validation Middleware

In [ ]:
import json, anthropic, logging
import instructor
from pydantic import BaseModel, Field
from typing import Literal, TypeVar, Generic, Callable
from functools import wraps

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("agent.schema")

T = TypeVar('T', bound=BaseModel)

class StructuredAgentCall(Generic[T]):
    """Production wrapper for schema-validated LLM calls."""
    
    def __init__(self, response_model: type[T], max_retries: int = 3):
        self.response_model = response_model
        self.max_retries = max_retries
        self.client = instructor.from_anthropic(anthropic.Anthropic())
        self.call_count = 0
        self.retry_count = 0
    
    def call(self, prompt: str, system: str = None, **kwargs) -> T | None:
        """Make a validated LLM call with automatic retry on schema failure."""
        self.call_count += 1
        messages = [{"role": "user", "content": prompt}]
        
        try:
            result = self.client.messages.create(
                model="claude-opus-4-5",
                max_tokens=1024,
                response_model=self.response_model,
                max_retries=self.max_retries,
                messages=messages,
                **kwargs
            )
            logger.info(f"✓ Call {self.call_count} succeeded")
            return result
        except Exception as e:
            logger.error(f"✗ Call {self.call_count} failed after {self.max_retries} retries: {e}")
            return None
    
    def stats(self) -> dict:
        return {
            "total_calls": self.call_count,
        }

# Define final output model
class FinalDeviceReport(BaseModel):
    device_id: str
    risk_level: Literal["LOW", "MED", "HIGH"]
    anomaly_score: float = Field(ge=0, le=1)
    root_cause: str
    actions: list[str] = Field(min_length=1)
    confidence: float = Field(ge=0, le=1)

# Run production-grade call
agent = StructuredAgentCall(FinalDeviceReport, max_retries=3)

result = agent.call(
    "Device JNP-001: CPU=94.2%, PFE errors=412/hr, BGP flaps=8. Provide structured analysis."
)

if result:
    print(f"\n✓ Production-grade structured output:")
    print(f"  Device: {result.device_id}")
    print(f"  Risk: {result.risk_level} (score: {result.anomaly_score:.2f})")
    print(f"  Confidence: {result.confidence:.0%}")
    print(f"  Cause: {result.root_cause}")
    print(f"  Actions ({len(result.actions)}): {result.actions[0]}...")
else:
    print("Analysis failed — check logs")

print(f"\nStats: {agent.stats()}")

---
## 🏆 Module Challenge

Build a **Schema-Validated Multi-Agent Classification System** that:

1. Defines a Pydantic model `AlertClassification` with fields:
   - `alert_type`: enum of `["hardware", "software", "network", "capacity"]`
   - `urgency`: enum of `["immediate", "within_hours", "within_days"]`
   - `ticket_priority`: int from 1-5
   - `auto_resolvable`: bool
   - `tags`: list of strings

2. Uses `pydantic_to_anthropic_tool()` to convert it to an Anthropic tool

3. Classifies 3 different alert messages using the tool

4. Validates each response with `Draft7Validator.iter_errors()`

5. Prints a summary table of classifications

**Bonus:** Add a cross-field validator: if `urgency=="immediate"`, then `ticket_priority` must be 1 or 2.

In [ ]:
# Your solution here!

# Step 1: Define AlertClassification model

# Step 2: Convert to Anthropic tool

# Step 3: Classify 3 alerts
alerts = [
    "JNP-001: CPU at 94%, PFE errors spiking to 400/hr",
    "JNP-002: Memory leak detected in rpd process, growing 50MB/hour",
    "JNP-003: BGP session with upstream peer flapping every 15 minutes",
]

# Step 4: Validate each response

# Step 5: Print summary table

---
## 📚 Module Summary

| Concept | Tool | Key Takeaway |
|---------|------|--------------|
| **Schema writing** | JSON Schema | Use `type`, `properties`, `required`, `enum`, `$ref` |
| **Validation** | `jsonschema` | Always `iter_errors()`, never `validate()` in prod |
| **LLM tool use** | `anthropic` | `tool_choice: {type: tool, name: ...}` forces output |
| **Pydantic** | `pydantic` | `model_json_schema()` auto-generates schemas |
| **Extraction** | `instructor` | `response_model=YourModel` — one line, full validation |

### Critical Rules
1. `description` fields = prompts to the LLM. Write them as instructions.
2. Always use `additionalProperties: false` on final output schemas.
3. Cap retries at 3. Log errors. Fail gracefully after that.
4. Use `Draft7Validator.iter_errors()` to collect ALL errors before retrying.
5. Prefer `instructor` for new projects — it handles the retry loop for you.

---
**Next Module → Module 03: XML & Markdown for Prompt Engineering**